In [5]:
from graphviz import Digraph

def create_panopticon_arch():
    dot = Digraph('MultiSensor_Panopticon', format='png')
    dot.attr(dpi='300', rankdir='TB', compound='true')
    dot.attr('node', fontname='Arial', shape='box', style='filled,rounded', fontsize='12')

    # --- 1. Multi-Sensor Input Stage (PE) ---
    with dot.subgraph(name='cluster_input') as c:
        c.attr(label='I. Sensor-Specific Patch Embedding', style='dashed', fontcolor='blue')
        sensors = [
            ('s2', 'Sentinel-2', '#CCEBC5'),
            ('l89', 'Landsat 8/9', '#B3CDE3'),
            ('s5p', 'TROPOMI (S5P)', '#DECBE4')
        ]
        for key, name, color in sensors:
            # 简化代码中的 Conv3d + ChnAttn + Proj
            c.node(f'PE_{key}', f'{name} PE\n(Conv3D + ChnAttn)', fillcolor=color)

    # --- 2. Shared Backbone (ViT Blocks) ---
    with dot.subgraph(name='cluster_backbone') as c:
        c.attr(label='II. Universal Backbone (DinoV2 Based)', style='filled', fillcolor='#F9F9F9')
        
        # 内部展示一个典型的 Block 结构，体现 DS-LN
        c.node('LN1', 'Domain-Specific LN 1\n(Selects s2/l89/s5p parameters)', fillcolor='#FFD1D1', color='red', penwidth='2')
        c.node('Attn', 'Shared Attention\n(Global Context)', fillcolor='#FFF2CC')
        c.node('LN2', 'Domain-Specific LN 2', fillcolor='#FFD1D1', color='red', penwidth='2')
        c.node('FFN_1', 'Domain-Specific SwiGLU FFN 1', fillcolor='#FFD1D1', color='red', penwidth='2')
        c.node('FFN_2', 'Shared SwiGLU FFN 2\n(Knowledge Base)', fillcolor='#FFF2CC')
        
        c.edge('LN1', 'Attn')
        c.edge('Attn', 'LN2')
        c.edge('LN2', 'FFN_1')
        c.edge('FFN_1', 'FFN_2')
        
        # 标注 12层 堆叠
        c.node('Stack', '... Repeat x12 Blocks ...', shape='none', style='')
        c.edge('FFN_2', 'Stack')

    # --- 3. Output Heads ---
    with dot.subgraph(name='cluster_heads') as c:
        c.attr(label='III. Task-Specific Heads', style='dashed', fontcolor='green')
        for key, name, color in sensors:
            c.node(f'Head_{key}', f'CLS Head ({name})\nMethane Yes/No', fillcolor=color)

    # --- 连接全局 ---
    for key, _, _ in sensors:
        dot.edge(f'PE_{key}', 'LN1', lhead='cluster_backbone')
        dot.edge('Stack', f'Head_{key}', ltail='cluster_backbone')

    dot.render('panopticon_final_arch', cleanup=True)
    print("架构图已生成：panopticon_final_arch.png")

if __name__ == "__main__":
    create_panopticon_arch()

架构图已生成：panopticon_final_arch.png


In [19]:
from graphviz import Digraph

def create_no_container_arch():
    # compound=true 仍然保留，以确保整体布局严谨
    dot = Digraph('Panopticon_No_Container', format='png')
    dot.attr(dpi='300', rankdir='TB', compound='true')
    dot.attr('node', fontname='Arial', shape='box', style='filled,rounded', fontsize='12')

    # --- 1. Sensor-Specific Input (PE) ---
    with dot.subgraph(name='cluster_input') as c:
        c.attr(label='I. Sensor-Specific Patch Embedding', style='dashed', fontcolor='#444444')
        sensors = [('s2', 'Sentinel-2', '#CCEBC5'), ('l89', 'Landsat 8/9', '#B3CDE3'), ('s5p', 'S5P (TROPOMI)', '#DECBE4')]
        for key, name, color in sensors:
            c.node(f'PE_{key}', f'{name} PE\n(3D Conv + ChnAttn)', fillcolor=color)

    # --- 2. Backbone (已经去掉了内部的 5x Adapter 容器) ---
    with dot.subgraph(name='cluster_backbone') as c:
        c.attr(label='II. Hybrid ViT Backbone', style='filled', fillcolor='#F9F9F9')
        
        c.node('Blocks_Shared', '7 x Shared Transformer Blocks\n(Universal Features)', fillcolor='#FFF2CC', width='4')
        
        # 直接定义原本在容器内的节点
        # 核心分叉点 (Input X)
        dot.node('Split', '', shape='point', width='0.1')
        
        # 中心主干道
        dot.node('Shared_Core', '5 x Shared NestedTensorBlock\n(Attn + MLP)', fillcolor='#FFF2CC', penwidth='2', width='3')
        
        # 侧边旁路适配器
        dot.node('Ad_S2', 'Tiny Adapter\n(Sentinel-2)', fillcolor='#CCEBC5', style='filled,dashed')
        dot.node('Ad_L89', 'Tiny Adapters\n(Landsat-8/9)', fillcolor='#B3CDE3', style='filled,dashed')
        dot.node('Ad_S5p', 'Tiny Adapters\n(Sentinel-5p)', fillcolor='#FFD1D1', style='filled,dashed')
        
        # 核心汇合点
        dot.node('Sum', 'Sum (+)', shape='circle', fillcolor='#D1FFD1')

        # 保持横向对齐逻辑
        with dot.subgraph() as s:
            s.attr(rank='same')
            dot.node('Ad_S2')
            dot.node('Shared_Core')
            dot.node('Ad_L89')
            dot.node('Ad_S5p')

        # --- 内部连线逻辑 ---
        # 增加从 Blocks_Shared 到 Split 的权重和距离
        dot.edge('Blocks_Shared', 'Split', minlen='1.2')
        
        # 主干直线
        dot.edge('Split', 'Shared_Core', weight='10')
        dot.edge('Shared_Core', 'Sum', weight='10')
        
        # 并行分叉
        dot.edge('Split', 'Ad_S2', style='dashed')
        dot.edge('Split', 'Ad_L89', style='dashed')
        dot.edge('Split', 'Ad_S5p', style='dashed')
        
        # 汇合
        dot.edge('Ad_S2', 'Sum', style='dashed')
        dot.edge('Ad_L89', 'Sum', style='dashed')
        dot.edge('Ad_S5p', 'Sum', style='dashed')

    # --- 3. Classifier Heads ---
    with dot.subgraph(name='cluster_heads') as c:
        c.attr(label='III. Specialized CLS Heads', style='dashed')
        c.node('H_S2', 'Head (Sentinel-2)', fillcolor='#CCEBC5')
        c.node('H_L89', 'Head (Landsat 8/9)', fillcolor='#B3CDE3')
        c.node('H_S5p', 'Head (Sentinel-5p)', fillcolor='#FFD1D1')

    # --- 全局连线 ---
    for key, _, _ in sensors:
        dot.edge(f'PE_{key}', 'Blocks_Shared')
    
    # 直接连向 Head
    dot.edge('Sum', 'H_S2')
    dot.edge('Sum', 'H_L89')
    dot.edge('Sum', 'H_S5p')

    dot.render('panopticon_no_container', cleanup=True)
    print("架构图已生成（去掉了 Adapter 容器）：panopticon_residual_adapter.png")

if __name__ == "__main__":
    create_no_container_arch()

架构图已生成（去掉了 Adapter 容器）：panopticon_residual_adapter.png


In [23]:
from graphviz import Digraph

def draw_balanced_adapter_block():
    dot = Digraph('SensorAdapterBlock_Balanced', format='png')
    dot.attr(dpi='300', rankdir='LR', compound='true')
    dot.attr('node', fontname='Arial', shape='box', style='filled,rounded', fontsize='12')

    # --- 输入节点：保持精简尺寸 ---
    dot.node('input', 'Input Feature\n(768-dim)', shape='circle', 
             fillcolor='#E1E1E1', width='0.8', height='0.8', fixedsize='true', fontsize='10')

    # --- 1. 主路径 (Shared Block) ---
    with dot.subgraph(name='cluster_main') as c:
        c.attr(label='Shared Block', style='filled', fillcolor='#F0F0F0')
        # block_core 现在会被 minlen 推向右侧
        c.node('block_core', 'Standard NestedTensorBlock\n(Attn + MLP + Internal Residuals)', fillcolor='#FFF2CC')

    # --- 2. 适配器路径 (Parallel Adapters) ---
    with dot.subgraph(name='cluster_adapters') as c:
        c.attr(label='Trainable Sensor Adapters', style='dashed', fontcolor='#CC0000')
        
        with dot.subgraph(name='cluster_bottleneck') as b:
            b.attr(label='TinyResidualAdapter Structure', style='dotted')
            b.node('down', 'Linear Down-proj\n(768 -> 16)', fillcolor='#FFD1D1')
            b.node('act', 'GELU Activation', shape='ellipse', fillcolor='#FFD1D1')
            b.node('up', 'Linear Up-proj\n(16 -> 768)', fillcolor='#FFD1D1')
            b.edge('down', 'act')
            b.edge('act', 'up')

    # --- 3. 融合阶段 ---
    dot.node('plus', 'Sum (+)', shape='circle', fillcolor='#D1FFD1', 
             width='0.5', height='0.5', fixedsize='true', fontsize='10')
    dot.node('output', 'Output Feature\n(768-dim)', shape='circle', 
             fillcolor='#E1E1E1', width='0.8', height='0.8', fixedsize='true', fontsize='10')

    # --- 关键连线调整 ---
    # 为主路径增加 minlen，使其向右移动，与旁路的 down 节点对齐起始位置或在视觉上更居中
    dot.edge('input', 'block_core', minlen='2.0') 
    
    # 旁路连线保持默认或微调
    dot.edge('input', 'down', label='Select via ModuleDict\n(e.g., S2, L8/9 or S5p)', fontsize='9')
    
    dot.edge('block_core', 'plus')
    dot.edge('up', 'plus')
    dot.edge('plus', 'output')

    dot.render('adapter_balanced_layout', cleanup=True)
    print("结构图已生成（Shared Block 已向右移动）：adapter_balanced_layout.png")

if __name__ == "__main__":
    draw_balanced_adapter_block()

结构图已生成（Shared Block 已向右移动）：adapter_balanced_layout.png
